In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import differential_evolution
from lifelines import CoxPHFitter
import warnings

In [ ]:
# Suppress lifelines convergence warnings during stochastic search
warnings.filterwarnings("ignore")

In [ ]:
# 1. Define the extended set of powers P
POWER_SET = [None, -3, -2.5, -2, -1.5, -1, -0.5, -0.25, 0, 0.25, 0.5, 1, 1.5, 2, 2.5, 3]

In [ ]:
def generate_fp_features(df, features, powers):
    transformed = {}

    for col, (p1, p2) in zip(features, powers):
        x = df[col].values

        for idx, p in enumerate(sorted([p for p in (p1, p2) if p is not None])):
            if p == 0:
                z = np.log(x)
            else:
                z = np.power(x, p)

            if idx == 1 and p == p1:
                z = z * np.log(x)

            transformed[f"{col}_fp{idx+1}_{p}"] = z

    return pd.DataFrame(transformed, index=df.index)

In [ ]:
# Global Cache for objective function
evaluation_cache = {}

def objective_function(de_vars, df, features, duration_col, event_col):
    indices = tuple(np.floor(de_vars).astype(int))

    # Cache check
    if indices in evaluation_cache:
        return evaluation_cache[indices]

    powers = []
    for i in range(len(features)):
        p1 = POWER_SET[indices[2*i]]
        p2 = POWER_SET[indices[2*i + 1]]
        powers.append((p1, p2))

    df_fp = generate_fp_features(df, features, powers)

    if df_fp.shape[1] == 0:
        return 1e10

    df_model = df_fp.copy()
    df_model[duration_col] = df[duration_col]
    df_model[event_col] = df[event_col]

    cph = CoxPHFitter(penalizer=0.01)

    try:
        cph.fit(df_model, duration_col=duration_col, event_col=event_col)

        n_events = max(df[event_col].sum(), 2)
        k = len(cph.params_)
        ll = cph.log_likelihood_

        bic = -2 * ll + k * np.log(n_events)

    except:
        bic = 1e10

    evaluation_cache[indices] = bic
    return bic

In [ ]:
class ConvergenceTracker:
    def __init__(self):
        self.best_val = np.inf
        self.history = []

    def evaluate(self, de_vars, df, features, duration_col, event_col):
        """Wraps the objective function to track the best seen value."""
        val = objective_function(de_vars, df, features, duration_col, event_col)
        if val < self.best_val:
            self.best_val = val
        return val

    def callback(self, xk, convergence=None):
        """Called by scipy.optimize at the end of every iteration."""
        self.history.append(self.best_val)

In [ ]:
def plot_convergence(history):
    """Plots the convergence curve of the DE optimization."""
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(history) + 1), history, marker='o', linestyle='-', color='b')
    plt.title('Differential Evolution Convergence for FP Power Selection')
    plt.xlabel('Generation / Iteration')
    plt.ylabel('Best BIC Score')
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
def preprocess_positive(df, features):
    """Ensures all continuous features are strictly positive for log/power transformations"""
    df_copy = df.copy()
    for col in features:
        x = df_copy[col].astype(float)
        if (x <= 0).any():
            df_copy[col] = x - x.min() + 1e-5
    return df_copy

In [ ]:
def select_fp_cox_de(df, features, duration_col, event_col, maxiter=50, popsize=15):
    dimensions = 2 * len(features)
    
    # FIXED BOUNDS: Must be 15.999 so np.floor can reliably select index 15
    bounds = [(0, len(POWER_SET) - 0.001)] * dimensions 
    
    print(f"Starting DE Optimization for {len(features)} features...")
    
    tracker = ConvergenceTracker()
    
    # FIXED: Removed workers=-1 and updating='deferred' to restore speed and caching
    result = differential_evolution(
        func=tracker.evaluate,
        bounds=bounds,
        args=(df, features, duration_col, event_col),
        maxiter=maxiter,
        popsize=popsize,
        mutation=(0.5, 1.0),
        recombination=0.7,
        seed=42,
        disp=True,
        callback=tracker.callback
    )
    
    best_indices = np.floor(result.x).astype(int)
    best_powers = []
    
    print("\n--- Optimal Power Selection ---")
    for i in range(len(features)):
        p1 = POWER_SET[best_indices[2*i]]
        p2 = POWER_SET[best_indices[2*i + 1]]
        best_powers.append((p1, p2))
        print(f"{features[i]}: Power 1 = {p1}, Power 2 = {p2}")
        
    print(f"Minimum BIC Achieved: {result.fun}")
    
    plot_convergence(tracker.history)
    
    return best_powers, result.fun

In [ ]:
# ==========================================
# Execution Block
# ==========================================
if __name__ == "__main__":
    np.random.seed(42)
    n = 300
    df = pd.DataFrame({
        'age': np.random.uniform(30, 80, n),
        'biomarker1': np.random.exponential(1.5, n),
        'biomarker2': np.random.normal(50, 10, n),
        'duration': np.random.randint(1, 100, n),
        'event': np.random.binomial(1, 0.7, n)
    })
    
    continuous_features = ['age', 'biomarker1', 'biomarker2']
    
    # 🔥 FIXED: Apply the preprocessing function BEFORE optimizing
    df_positive = preprocess_positive(df, continuous_features)
    
    best_powers, best_bic = select_fp_cox_de(
        df=df_positive, # Pass the positive dataset 
        features=continuous_features, 
        duration_col='duration', 
        event_col='event',
        maxiter=20,  
        popsize=10
    )
    
    # Use the positive dataset for the final model generation
    final_df = generate_fp_features(df_positive, continuous_features, best_powers)
    final_df['duration'] = df_positive['duration']
    final_df['event'] = df_positive['event']
    
    final_cph = CoxPHFitter(penalizer=0.01)
    final_cph.fit(final_df, duration_col='duration', event_col='event')
    print("\n--- Final Model Summary ---")
    final_cph.print_summary()

Starting DE Optimization for 3 features...


In [ ]:
# ==========================================
# Comparison
# ==========================================
print("Fitting Traditional Cox Model (Linear Covariates)")
print("\n" + "="*50)
print("="*50)
    
traditional_df = df_positive[continuous_features + ['duration', 'event']].copy()
    
traditional_cph = CoxPHFitter(penalizer=0.01)
traditional_cph.fit(traditional_df, duration_col='duration', event_col='event')
    
n_events_trad = traditional_df['event'].sum()
if n_events_trad <= 1: 
    n_events_trad = len(traditional_df)
        
k_trad = len(traditional_cph.params_)
ll_trad = traditional_cph.log_likelihood_
    
traditional_bic = -2 * ll_trad + k_trad * np.log(n_events_trad)
    
print("\n--- Traditional Model Summary ---")
traditional_cph.print_summary()
    
print("\n" + "="*50)
print("MODEL COMPARISON RESULTS")
print("="*50)
    
fp_c_index = final_cph.concordance_index_
trad_c_index = traditional_cph.concordance_index_
print(f"{'Metric':<20} | {'Traditional Cox':<18} | {'FP Cox (DE Optimized)':<20}")
print("-" * 65)
print(f"{'Log-Likelihood':<20} | {ll_trad:<18.4f} | {final_cph.log_likelihood_:<20.4f}")
print(f"{'BIC (Lower is better)':<20} | {traditional_bic:<18.4f} | {best_bic:<20.4f}")
print(f"{'Concordance Index':<20} | {trad_c_index:<18.4f} | {fp_c_index:<20.4f}")
print("-" * 65)
if best_bic < traditional_bic:
    print("\nConclusion: The Fractional Polynomial model (via DE) achieved a LOWER BIC, indicating a better fit.")
else:
    print("\nConclusion: The Traditional Cox model achieved a LOWER BIC.")


Fitting Traditional Cox Model (Linear Covariates)

--- Traditional Model Summary ---


<lifelines.CoxPHFitter: fitted with 300 total observations, 91 right-censored observations>
             duration col = 'duration'
                event col = 'event'
                penalizer = 0.01
                 l1 ratio = 0.0
      baseline estimation = breslow
   number of observations = 300
number of events observed = 209
   partial log-likelihood = -993.70
         time fit was run = 2026-02-22 07:47:56 UTC

---
            coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                   
age        -0.00      1.00      0.00           -0.01            0.01                0.99                1.01
biomarker1 -0.02      0.98      0.04           -0.11            0.06                0.90                1.06
biomarker2 -0.01      0.99      0.01           -0.02            0.01                0.98                1.01

            cmp to     z    p  -log2(p)
covariate                              
age           0.00 -0.38 0.71      0.50
biomarker1    0.00 -0.50 0.62      0.69
biomarker2    0.00 -1.19 0.23      2.10
---
Concordance = 0.52
Partial AIC = 1993.40
log-likelihood ratio test = 1.73 on 3 df
-log2(p) of ll-ratio test = 0.67


MODEL COMPARISON RESULTS
Metric               | Traditional Cox    | FP Cox (DE Optimized)
-----------------------------------------------------------------
Log-Likelihood       | -993.7005          | -993.5106           
BIC (Lower is better) | 2003.4280          | 2008.3906           
Concordance Index    | 0.5210             | 0.5191              
-----------------------------------------------------------------

Conclusion: The Traditional Cox model achieved a LOWER BIC. The non-linear transformations did not justify the extra complexity on this dataset.
